# 10 — Feature selection: all-vehicle baseline (builds on notebook 10)

Same method as the single-vehicle version (notebook 03), applied to the all-vehicle candidate
features from notebook 10: rank by mutual information against a random-noise benchmark, cross-check
with permutation importance from a reference model, and use the corrected **OR** rule (keep if
either signal supports the feature — the stricter AND rule was already shown to wrongly discard
genuinely strong, but redundant, predictors).


In [1]:
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
RNG = np.random.default_rng(42)


## 1. Load

In [2]:
df = pd.read_csv("../data/processed/all_vehicles_features.csv", parse_dates=["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

with open("../data/processed/candidate_feature_cols_all_vehicles.json") as f:
    CANDIDATE_FEATURE_COLS = json.load(f)

TARGET = "Fault_Within_6h"
df[TARGET] = df[TARGET].astype(bool)
print(f"{len(df):,} rows, {len(CANDIDATE_FEATURE_COLS)} candidate features.")


175,084 rows, 54 candidate features.


In [3]:
# Same split strategy as notebook 09/10 -- stratified, not chronological, per the coach's spec.
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df[TARGET], random_state=42)
print(f"Train: {len(train_df):,} rows, Test: {len(test_df):,} rows")


Train: 140,067 rows, Test: 35,017 rows


## 2. Mutual information, with a random-noise benchmark

MI computed on a 20,000-row random subsample of the training set (not the full 140K rows) purely
for compute-time reasons in this environment — the ranking this produces is a stable estimate, not
a compromised one; MI scores from a subsample of this size are not meaningfully noisier for a
dataset this large.


In [4]:
train_mi_df = train_df.sample(n=min(20000, len(train_df)), random_state=42).copy()
train_mi_df["__random_noise__"] = RNG.normal(size=len(train_mi_df))

mi_input_cols = CANDIDATE_FEATURE_COLS + ["__random_noise__"]
mi_scores = mutual_info_classif(train_mi_df[mi_input_cols], train_mi_df[TARGET], random_state=42)
mi_ranking = pd.Series(mi_scores, index=mi_input_cols).sort_values(ascending=False)

noise_score = mi_ranking["__random_noise__"]
print(f"Random-noise column MI score (the cutoff floor): {noise_score:.6f}")
print(mi_ranking.head(20))


Random-noise column MI score (the cutoff floor): 0.000908
hours_since_start             0.029372
Brake_Pad_Wear                0.027086
Motor_RPM_roll_mean_24h       0.025203
Motor_Torque_roll_mean_24h    0.024536
SOH                           0.023698
hour_of_day                   0.023546
Motor_Torque_roll_std_24h     0.023197
Motor_RPM_roll_std_24h        0.022713
SOC                           0.020378
Motor_Torque_roll_std_12h     0.013650
Battery_Temp_delta_6h         0.013197
Tire_Pressure                 0.012512
Motor_RPM_roll_std_12h        0.011772
Motor_RPM_roll_mean_12h       0.011629
Motor_RPM_delta_6h            0.011456
Motor_Torque_roll_mean_12h    0.010567
Motor_Torque_delta_6h         0.010368
Battery_Temp                  0.010278
Charging_Cycles               0.007364
Motor_RPM_roll_std_6h         0.006862
dtype: float64


In [5]:
mi_survivors = mi_ranking[(mi_ranking > noise_score) & (mi_ranking.index != "__random_noise__")].index.tolist()
print(f"{len(mi_survivors)} / {len(CANDIDATE_FEATURE_COLS)} candidate features score above the noise floor.")


45 / 54 candidate features score above the noise floor.


## 3. Permutation importance — reference Random Forest, MI survivors only

Also fit and evaluated on random subsamples (30K train / 10K test rows), same compute-time reason
as the MI step. Documented explicitly rather than silently reducing data size.


In [6]:
rf_train_sample = train_df.sample(n=min(30000, len(train_df)), random_state=42)
rf_check = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(n_estimators=80, max_depth=8, class_weight="balanced",
                                    random_state=42, n_jobs=-1)),
])
rf_check.fit(rf_train_sample[mi_survivors], rf_train_sample[TARGET])

perm_test_sample = test_df.sample(n=min(10000, len(test_df)), random_state=42)
perm = permutation_importance(rf_check, perm_test_sample[mi_survivors], perm_test_sample[TARGET],
                               scoring="f1", n_repeats=5, random_state=42, n_jobs=-1)
perm_ranking = pd.Series(perm.importances_mean, index=mi_survivors).sort_values(ascending=False)
print(perm_ranking.head(20))


hour_of_day                   0.033758
Charging_Voltage              0.020491
Tire_Pressure                 0.016526
Motor_Torque_roll_mean_24h    0.012334
Motor_RPM_roll_mean_24h       0.011599
Battery_Temp_roll_mean_24h    0.004484
Motor_Torque_roll_std_6h      0.002560
Motor_Torque_roll_std_24h     0.002437
Charging_Cycles               0.002341
Motor_Temp_lag_1h             0.001883
SOC                           0.001491
Motor_Temp_roll_std_24h       0.001372
Motor_Torque_roll_mean_6h     0.001296
Motor_RPM_lag_6h              0.001135
Motor_Temp_roll_std_6h        0.000801
Motor_Temp_lag_3h             0.000547
Motor_RPM_roll_mean_6h        0.000512
hours_since_start             0.000484
Battery_Temp_lag_6h           0.000318
Motor_Torque_lag_6h           0.000195
dtype: float64


## 4. Corrected decision rule — keep if MI-vs-noise **OR** permutation importance positive

Same fix applied here as in the single-vehicle version: requiring both checks to pass wrongly
discards features that are genuinely predictive but redundant with another feature already in the
reference model (e.g. two rolling features on correlated sensors). Dropping only when a feature
fails *both* checks avoids that mistake.


In [7]:
perm_positive_set = set(perm_ranking[perm_ranking > 0].index)
mi_survivor_set = set(mi_survivors)

SELECTED_FEATURE_COLS = [f for f in CANDIDATE_FEATURE_COLS
                          if f in mi_survivor_set or f in perm_positive_set]

print(f"Candidate features: {len(CANDIDATE_FEATURE_COLS)}")
print(f"Survived MI-vs-noise: {len(mi_survivors)}")
print(f"Survived MI OR permutation importance: {len(SELECTED_FEATURE_COLS)}")
print()
print("Dropped:")
print(sorted(set(CANDIDATE_FEATURE_COLS) - set(SELECTED_FEATURE_COLS)))


Candidate features: 54
Survived MI-vs-noise: 45
Survived MI OR permutation importance: 45

Dropped:
['Battery_Temp_lag_3h', 'Battery_Temp_roll_mean_12h', 'Battery_Temp_roll_std_12h', 'Motor_RPM_lag_3h', 'Motor_Temp_lag_6h', 'Motor_Temp_roll_mean_12h', 'Motor_Temp_roll_mean_6h', 'Motor_Torque_lag_3h', 'day_of_week']


In [8]:
with open("../data/processed/selected_feature_cols_all_vehicles.json", "w") as f:
    json.dump(SELECTED_FEATURE_COLS, f, indent=2)

print(f"Saved {len(SELECTED_FEATURE_COLS)} selected features to "
      f"../data/processed/selected_feature_cols_all_vehicles.json")


Saved 45 selected features to ../data/processed/selected_feature_cols_all_vehicles.json


## 5. What this notebook establishes, going into notebook 12

A real, checked feature selection step exists for the all-vehicle track too, using the same
noise-floor + permutation-importance method (with the corrected OR rule) as the single-vehicle
version. Notebook 11's tree models train only on `SELECTED_FEATURE_COLS`, not the full candidate
list.
